In [7]:
# Import

import os
import glob
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm


# TensorFlow
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image


# PyTorch / DINOv2
import torch
from torchvision import transforms
import joblib

In [8]:
# Racine du dataset
dataset_root = "images/Images"

# Fichiers modèles
MOBILENET_PATH = "best_mobilenetv2_finetuned.keras"
DINOV2_CLF_PATH = "best_dinov2_classifier.keras"

# Sortie
OUTPUT_CSV = "predictions_stanford_dogs.csv"

# Sécurité CPU
torch.set_num_threads(1)

In [9]:
classes = sorted([
d for d in os.listdir(dataset_root)
if os.path.isdir(os.path.join(dataset_root, d))
])

print(f"{len(classes)} classes détectées")

120 classes détectées


In [10]:
image_paths = []
labels = []

for cls in classes:
    cls_folder = os.path.join(dataset_root, cls)

    imgs = glob.glob(os.path.join(cls_folder, "*.jpg"))
    imgs += glob.glob(os.path.join(cls_folder, "*.jpeg"))
    imgs += glob.glob(os.path.join(cls_folder, "*.png"))

    for img_path in imgs:
        image_paths.append(img_path)
        labels.append(cls)

print(f"{len(image_paths)} images trouvées")

20580 images trouvées


MobileNetv2

In [11]:
mobilenet = load_model(MOBILENET_PATH, compile=False)
print("MobileNetV2 chargé")

MobileNetV2 chargé


In [12]:
def predict_mobilenet(model, img):
    target_size = (model.input_shape[1], model.input_shape[2])
    img_resized = img.resize(target_size)

    x = image.img_to_array(img_resized)
    x = np.expand_dims(x, axis=0) / 255.0

    preds = model.predict(x, verbose=0)
    idx = int(np.argmax(preds, axis=1)[0])

    return classes[idx], float(preds[0][idx])

Dinov2

In [13]:
dinov2_backbone = torch.hub.load(
    "facebookresearch/dinov2",
    "dinov2_vitb14",
    pretrained=True
)
dinov2_backbone.eval()

Using cache found in C:\Users\maely/.cache\torch\hub\facebookresearch_dinov2_main


DinoVisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 768, kernel_size=(14, 14), stride=(14, 14))
    (norm): Identity()
  )
  (blocks): ModuleList(
    (0-11): 12 x NestedTensorBlock(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (attn): MemEffAttention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (proj): Linear(in_features=768, out_features=768, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (drop_path1): Identity()
      (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=3072, out_features=768, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
      (ls2): LayerScale()
      (drop_path2): Identity()
    )
  )
  (norm): LayerNorm((768,), eps=1e-06, elementwise_affi

In [14]:
dinov2_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [15]:
def predict_dinov2(backbone, clf, img):
    x = dinov2_transform(img).unsqueeze(0)

    with torch.no_grad():
        emb = backbone(x)
        emb = emb.cpu().numpy().reshape(1, -1)

    preds = clf.predict(emb, verbose=0)
    idx = int(np.argmax(preds, axis=1))

    return classes[idx], float(preds[0][idx])

Prédictions

In [18]:
from tensorflow.keras.models import load_model

DINOV2_CLF_PATH = "best_dinov2_classifier.keras"

dinov2_clf = load_model(DINOV2_CLF_PATH)
print("Classifieur DINOv2 chargé")

Classifieur DINOv2 chargé


In [20]:
img_test = Image.open(image_paths[0]).convert("RGB")

print("MobileNet:", predict_mobilenet(mobilenet, img_test))
print("DINOv2:", predict_dinov2(dinov2_backbone, dinov2_clf, img_test))


MobileNet: ('n02085620-Chihuahua', 0.7826482057571411)
DINOv2: ('n02085620-Chihuahua', 0.9837360978126526)


C:\Users\maely\AppData\Local\Temp\ipykernel_28812\2213836879.py:9: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  idx = int(np.argmax(preds, axis=1))


In [21]:
records = []

for path, true_cls in tqdm(
    zip(image_paths, labels),
    total=len(image_paths),
    desc="Prédictions batch"
):
    img = Image.open(path).convert("RGB")

    # --- MobileNet ---
    mn_pred, mn_prob = predict_mobilenet(mobilenet, img)

    # --- DINOv2 (vitb14 + flatten) ---
    dn_pred, dn_prob = predict_dinov2(
        dinov2_backbone,
        dinov2_clf,
        img
    )

    records.append({
        "image_path": path,
        "true_class": true_cls,
        "mobilenet_pred": mn_pred,
        "mobilenet_proba": float(mn_prob),
        "dinov2_pred": dn_pred,
        "dinov2_proba": float(dn_prob),
    })

Prédictions batch:   0%|          | 0/20580 [00:00<?, ?it/s]C:\Users\maely\AppData\Local\Temp\ipykernel_28812\2213836879.py:9: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  idx = int(np.argmax(preds, axis=1))
Prédictions batch: 100%|██████████| 20580/20580 [4:09:14<00:00,  1.38it/s]  


In [22]:
df_preds = pd.DataFrame(records)

df_preds.to_csv(
    "predictions_mobilenet_dinov2.csv",
    index=False
)

print("CSV généré :", df_preds.shape)

CSV généré : (20580, 6)
